In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import re
import time

In [ ]:
# dataset: https://zenodo.org/records/17266923
# dataset repo: https://github.com/vintagedon/steam-dataset-2025/tree/main

In [ ]:
def get_steam_game_details(app_id, retries=10, backoff_factor=1.0):
    url = "https://store.steampowered.com/api/appdetails"
    params = {
        "appids": app_id,
        "l": "english"
    }
    headers = {"Accept-Language": "en-US,en;q=0.9"}

    for attempt in range(1, retries + 1):
        response = requests.get(url, params=params, headers=headers)

        if response.status_code == 200:
            break

        if response.status_code == 429 and attempt < retries:
            delay = backoff_factor * (2 ** (attempt - 1))
            print(f"429 received, backing off for {delay:.1f} seconds (attempt {attempt}/{retries})")
            time.sleep(delay)
            continue

        raise Exception(f"Request failed with status code {response.status_code}")

    data = response.json()

    # return data
    if not data[str(app_id)]["success"]:
        raise Exception("Failed to fetch game data")

    game_data = data[str(app_id)]["data"]

    return {
        "name": game_data.get("name"),
        "type": game_data.get("type"),
        "release_date": game_data.get("release_date", {}).get("date"),
        "price": game_data.get("price_overview", {}).get("final_formatted"),
        "genres": [g["description"] for g in game_data.get("genres", [])],
        "description": game_data.get("short_description"),
        "pc_requirements": game_data.get("pc_requirements"),
        "mac_requirements": game_data.get("mac_requirements"),
        "linux_requirements": game_data.get("linux_requirements"),
    }

In [ ]:
details = get_steam_game_details(570) #returns a dictionary with strings straight from JSON
print(details)

In [ ]:
from bs4 import BeautifulSoup

def parse_requirements_html(html_string):
    """Parse a single HTML requirements block into a dict."""
    if not html_string:
        return {}

    soup = BeautifulSoup(html_string, "html.parser")
    result = {}

    for li in soup.find_all("li"):
        strong = li.find("strong")
        if not strong:
            continue

        key = strong.get_text(strip=True).replace(":", "").replace("*", "").strip()
        strong.extract()  # remove label
        value = li.get_text(" ", strip=True)

        result[key] = value

    return result


def extract_system_requirements(game_data):
    """
    Takes the 'data' field from Steam API response and parses
    pc/mac/linux requirements into structured dicts.
    """
    systems = ["pc_requirements", "mac_requirements", "linux_requirements"]
    parsed = {}

    for system in systems:
        system_data = game_data.get(system)
        if not system_data:
            continue

        parsed[system] = {}

        for level in ["minimum", "recommended"]:
            html = system_data.get(level)
            if html:
                parsed[system][level] = parse_requirements_html(html)

    return parsed


In [ ]:
def parse_storage(app_id): 
    gamedata = get_steam_game_details(app_id)
    reqs = extract_system_requirements(gamedata)
    min_storage = -1
    rec_storage = -1
    keywords = ["storage", "hard", "drive", "disk", "space"]

    for system in reqs:
        for level in reqs[system]:
            level_data = reqs[system][level]
            storage_key = None
            for key in level_data:
                key_lower = key.lower()
                if any(term in key_lower for term in keywords):
                    storage_key = key
                    break

            if not storage_key:
                continue

            storage_str = str(level_data[storage_key]).lower()
            # Extract numeric part
            try:
                ##if size is in gb, then mb, then kb (hopefully no b)
                matches = re.findall(r'(\d+(?:\.\d+)?)\s*(?:gb|gigabytes?)', storage_str)
                if matches:
                    if "min" in level:
                        min_storage = float(matches[0])
                    elif "rec" in level:
                        rec_storage = float(matches[0])
                else:
                    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(?:mb|megabytes?)', storage_str)
                    if matches:
                        if "min" in level:
                            min_storage = float(matches[0]) * 0.001
                        elif "rec" in level:
                            rec_storage = float(matches[0]) * 0.001
                    else:
                        matches = re.findall(r'(\d+(?:\.\d+)?)\s*(?:kb|kilobytes?)', storage_str)
                        if matches:
                            if "min" in level:
                                min_storage = float(matches[0]) * 0.001 * 0.001
                            elif "rec" in level:
                                rec_storage = float(matches[0]) * 0.001 * 0.001
            except ValueError:
                continue

            break
    return min_storage, rec_storage

In [ ]:
# Example usage

parsed = extract_system_requirements(details)
print(parsed)
# import pprint
# pprint.pprint(parsed)

In [ ]:
#testing
min_storage, rec_storage = parse_storage(3987950)
print(f"Minimum Storage: {min_storage} GB")
print(f"Recommended Storage: {rec_storage} GB")

In [ ]:
df = pd.read_csv("steam_dataset_2025_csv/applications.csv")
df.columns

In [ ]:
df.tail(50)

In [ ]:
#filter for just games
df = df[
    (df['type'] == 'game') &
    (df['name'].notna()) &
    (df['name'].str.strip() != '') &
    (df['release_date'].notna())
]

bad_terms = [
    'soundtrack',
    'dedicated server',
    'demo',
    'test server',
    'benchmark'
]

mask = ~df['name'].str.lower().str.contains('|'.join(bad_terms), na=False)
df = df[mask]
print(df.shape)

In [ ]:
#do not run this cell again, it will overwrite the csv file with an empty dataframe
# storage_df = pd.DataFrame(columns=["app_id", "min_storage", "rec_storage"])
# storage_df.to_csv("storage_requirements.csv", index=False)

In [ ]:
storage_df = pd.read_csv("storage_requirements.csv")
i = 0
print(i)

In [ ]:
import time

attempt = 0
while i < df.shape[0]:
    app_id = df.iloc[i]["appid"]
    print(f"Processing app_id {app_id} ({i}/{df.shape[0]})")
    try:
        min_storage, rec_storage = parse_storage(app_id)
        print(min_storage, rec_storage)
        new_row = pd.DataFrame([{
            "app_id": app_id,
            "min_storage": min_storage,
            "rec_storage": rec_storage
        }])

        storage_df = pd.concat([storage_df, new_row], ignore_index=True)
    except Exception as e:
        print(f"Error processing app_id {app_id}: {e}")
        attempt += 1
        if attempt >= 3:
            print(f"Skipping app_id {app_id} after 3 attempts")
            attempt = 0
            i += 1
            continue
        continue

    time.sleep(1.0) # avoid hitting rate limits
    if i % 10 == 0:
        storage_df.to_csv("storage_requirements.csv", index=False)
    
    i += 1

In [ ]:
files = ['storage_requirements_1.csv', 'storage_requirements_2.csv']

#stack files
dfs = [pd.read_csv(file) for file in files]
storage_df = pd.concat(dfs, ignore_index=True)

df = df.merge(storage_df, left_on='appid', right_on='app_id', how='left')

df.drop(columns=['app_id'], inplace=True)



#df has the new columns: min_storage and rec_storage
print(df.head())

In [ ]:
df.head(20)

In [ ]:
#replace -1 with NaN for the storage columns
df['min_storage'] = df['min_storage'].replace(-1, np.nan)
df['rec_storage'] = df['rec_storage'].replace(-1, np.nan)

In [ ]:
df.to_csv("extended_dataset.csv", index=False)